<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/aiagentprojectmioipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
import re
from typing import List, Dict, Any
from dataclasses import dataclass, asdict
from datetime import datetime

@dataclass
class Task:
    id: str
    description: str
    assigned_to: str = None
    status: str = "pending"
    result: Any = None
    dependencies: List[str] = None

    def __post_init__(self):
        if self.dependencies is None:
            self.dependencies = []

@dataclass
class Agent:
    name: str
    role: str
    expertise: str
    system_prompt: str

AGENT_REGISTRY = {
    "researcher": Agent(name="researcher", role="Research Specialist", expertise="General information gathering", system_prompt="You are a research specialist."),
    "medical_researcher": Agent(name="medical_researcher", role="Medical Scientist", expertise="Deep medical research, pathology analysis, and clinical studies", system_prompt="You are a senior medical researcher. Provide evidence-based medical insights."),
    "medline_researcher": Agent(name="medline_researcher", role="Bibliographic Specialist", expertise="Searching Medline/PubMed for clinical evidence and literature reviews", system_prompt="You are a bibliographic specialist. Extract key study findings and provide PMID links."),
    "fact_checker": Agent(name="fact_checker", role="Fact-Checking Specialist", expertise="Verifying medical and scientific claims for accuracy and hallucinations", system_prompt="You are a fact-checking specialist. Identify any false claims, hallucinations, or unsupported dates/names in the provided text."),
    "therapy_specialist": Agent(name="therapy_specialist", role="Clinical Therapist", expertise="Specialized therapy protocols and treatment planning", system_prompt="You are a therapy specialist. Design safe and effective treatment plans."),
    "writer": Agent(name="writer", role="Content Writer", expertise="Clear communication", system_prompt="You are a professional writer.")
}

class LocalLLM:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
        if self.tokenizer.pad_token is None: self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate(self, prompt, max_tokens=300):
        inputs = self.tokenizer(f"<|user|>\n{prompt}</s>\n<|assistant|>", return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_tokens)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).split("<|assistant|>")[-1].strip()

class ManagerAgent:
    def __init__(self):
        self.llm = LocalLLM()
        self.agents = AGENT_REGISTRY
        self.tasks = {}
        self.log_book = []

    def log(self, m):
        print(f"[*] {m}")
        self.log_book.append(m)

    def search_medline(self, query):
        self.log(f"Ricerca Medline per: {query}")
        # Simulazione di risultati bibliografici
        results = [
            {"title": "Methotrexate-induced macrocytic anemia: a clinical review", "link": "https://pubmed.ncbi.nlm.nih.gov/1234567/", "summary": "Analysis of folate metabolism inhibition."},
            {"title": "Circadian rhythm diagnostics in clinical practice", "link": "https://pubmed.ncbi.nlm.nih.gov/7654321/", "summary": "Role of actigraphy and MSLT in sleep disorders."}
        ]
        return results

    def verify_fact(self, content):
        self.log("Avvio verifica dei fatti...")
        checker = self.agents['fact_checker']
        prompt = f"{checker.system_prompt}\n\nReview this content and list potential errors:\n{content}\n\nAnalysis:"
        return self.llm.generate(prompt)

    def execute_goal(self, goal):
        self.log(f"Executing: {goal}")
        bib_results = self.search_medline(goal)
        research_output = f"Trovati {len(bib_results)} studi. Esempio: {bib_results[0]['title']}"
        fact_check_report = self.verify_fact(research_output)
        return {"original": research_output, "bibliography": bib_results, "verification": fact_check_report}

In [6]:
def analyze_medical_agent_logic():
    manager = ManagerAgent()
    # Recuperiamo l'agente specifico
    med_agent = AGENT_REGISTRY['medical_researcher']

    # Simuliamo un task di analisi per il caso clinico precedente
    task = Task(
        id='t1',
        description='Analizza il meccanismo fisiopatologico dell\'anemia macrocitica in un paziente che assume metotrexato per artrite reumatoide.',
        assigned_to='medical_researcher'
    )

    print(f"--- Analisi Logica Agente: {med_agent.name} ---")
    print(f"Expertise: {med_agent.expertise}")
    print(f"System Prompt: {med_agent.system_prompt}\n")

    # Generazione della risposta specifica dell'agente
    prompt = f"{med_agent.system_prompt}\n\nTask: {task.description}\n\nAnalisi dettagliata:"
    response = manager.llm.generate(prompt)

    print("Risposta elaborata dall'agente:")
    print(response)

analyze_medical_agent_logic()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- Analisi Logica Agente: medical_researcher ---
Expertise: Deep medical research, pathology analysis, and clinical studies
System Prompt: You are a senior medical researcher. Provide evidence-based medical insights.

Risposta elaborata dall'agente:
Anemia macrocitica è un disturbo fisiopatologico che si verifica quando la circolazione sanguigna non riesce a trasportare abbastanza sangue da supportare la produzione di eritrociti. Questo disturbo è caratterizzato da una riduzione della produzione di eritrociti e da una riduzione della produzione di eritrociti di tipo megakariocitico (M-erythrocytes).

Il meccanismo fisiopatologico dell'anemia macrocitica è legato alla perdita di eritrociti di tipo megakariocitico, che sono i più grandi e più veloci. Questi eritrociti sono responsabili della produzione di eritrociti di tipo megakariocitico, che sono i più grandi e più veloci. Questi eritrociti sono in grado di trasportare più sangue rispetto ai eritrociti di tipo megakariocitico, che so

In [7]:
def answer_circadian_query():
    manager = ManagerAgent()
    med_agent = AGENT_REGISTRY['medical_researcher']

    question = """Quale metodo diagnostico può essere utilizzato per valutare i disturbi del ritmo circadiano?
Options: Test di latenza multipla del sonno (MSLT), Dosaggio della serotonina, Polisonnografia, Attimetria, Ecoencefalogramma"""

    prompt = f"{med_agent.system_prompt}\n\nDomanda: {question}\n\nFornisci la risposta corretta e una breve spiegazione scientifica:"
    response = manager.llm.generate(prompt, max_tokens=300)

    print(f"--- Risposta Medica Specialistica ---\n")
    print(response)

answer_circadian_query()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- Risposta Medica Specialistica ---

L'esame di latenza multipla del sonno (MSLT) è un metodo diagnostico che consente di valutare il ritmo circadiano. Questo metodo è basato sul fatto che il ritmo circadiano è correlato alla funzione cardiaca e alla funzione respiratoria. Il MSLT consente di misurare la durata e la frequenza dei periodi di somnolenza e di vigilia, che sono caratteristici del ritmo circadiano. Questo metodo è utile per determinare se un individuo ha un disturbo del ritmo circadiano, come il disturbo di insomnia, che può essere causato da un disturbo del ritmo circadiano.

Il test di latenza multipla del sonno (MSLT) è stato sviluppato da un gruppo di ricercatori del Massachusetts Institute of Technology (MIT) nel 1999. Questo test è stato approvato dall'American Psychiatric Association (APA) nel 2004 e dal Consiglio di medicina del Regno Unito nel 2005.

Il Dosaggio della serotonina (5-HT) è un metodo diagnostico che consente di valutare il ritmo circadiano. Questo m

In [9]:
def test_fact_checker():
    manager = ManagerAgent()
    # Testiamo la logica di verifica su un'affermazione sospetta
    goal = "Verifica validità test MSLT/MIT"
    results = manager.execute_goal(goal)

    print(f"--- Output Originale ---\n{results['original']}\n")
    print(f"--- Report di Verifica (Fact Checker) ---\n{results['verification']}")

test_fact_checker()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[*] Executing: Verifica validità test MSLT/MIT
[*] Avvio verifica dei fatti...
--- Output Originale ---
Il test MSLT è stato inventato dal MIT nel 1999 per il ritmo circadiano.

--- Report di Verifica (Fact Checker) ---
The provided text contains several false claims, hallucinations, and unsupported dates/names.

False claims:
- The MIT MSLT (Mind-Sweep Light Therapy) was invented in 1999. This is a misleading statement as the MSLT was invented in 1994 by Dr. Robert S. Langer and Dr. Robert J. Langer, who are not affiliated with MIT.

Hallucinations:
- The text claims that the "Mind-Sweep Light Therapy" is a "new treatment for depression." This is not true, as the MSLT has been used for over 20 years and is a proven treatment for depression.

Unsupported dates/names:
- The text claims that the "Mind-Sweep Light Therapy" was invented by Dr. Robert S. Langer and Dr. Robert J. Langer, but the text does not provide any evidence to support this claim.

Overall, the provided text contains se

In [11]:
def test_medline_agent():
    manager = ManagerAgent()
    goal = "Meccanismi di anemia da metotrexato"
    results = manager.execute_goal(goal)

    print(f"--- Risultati Bibliografici ---")
    for paper in results['bibliography']:
        print(f"Titolo: {paper['title']}")
        print(f"Link: {paper['link']}")
        print(f"Sintesi: {paper['summary']}\n")

    print(f"--- Validazione Fact Checker ---\n{results['verification']}")

test_medline_agent()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[*] Executing: Meccanismi di anemia da metotrexato
[*] Ricerca Medline per: Meccanismi di anemia da metotrexato
[*] Avvio verifica dei fatti...
--- Risultati Bibliografici ---
Titolo: Methotrexate-induced macrocytic anemia: a clinical review
Link: https://pubmed.ncbi.nlm.nih.gov/1234567/
Sintesi: Analysis of folate metabolism inhibition.

Titolo: Circadian rhythm diagnostics in clinical practice
Link: https://pubmed.ncbi.nlm.nih.gov/7654321/
Sintesi: Role of actigraphy and MSLT in sleep disorders.

--- Validazione Fact Checker ---
The provided text, Trovati 2 studi. Esempio: Methotrexate-induced macrocytic anemia: a clinical review, contains false claims, hallucinations, and unsupported dates/names.

False claims:
- The text states that "Methotrexate-induced macrocytic anemia: a clinical review" is an example of a study published in Trovati. However, the provided text does not provide any information about the specific study or its publication date.

Hallucinations:
- The text claims t

In [12]:
def analyze_hypertension_query():
    manager = ManagerAgent()
    med_agent = AGENT_REGISTRY['medical_researcher']

    question = """Quale dei seguenti risultati rilevati all'esame obiettivo o agli esami ematochimici di un paziente affetto da ipertensione arteriosa non suggerisce la presenza di ipertensione secondaria?
Options: Edemi ansarcatici, Palpitazioni, Soffio sistolico, Presenza di quarto tono (S4), Ipokaliemia"""

    prompt = f"{med_agent.system_prompt}\n\nAnalizza la domanda medica e identifica l'opzione corretta spiegando perché non suggerisce ipertensione secondaria:\n{question}"
    response = manager.llm.generate(prompt, max_tokens=350)

    print(f"--- Analisi Ipertensione Secondaria ---\n")
    print(response)

analyze_hypertension_query()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- Analisi Ipertensione Secondaria ---

Sure, I'd be happy to provide evidence-based medical insights.

1. A patient with hypertension who undergoes an echocardiogram and an angiogram does not suggest the presence of ipertensione secondaria.

The echocardiogram and angiogram are non-invasive diagnostic tests that can detect the presence of heart disease, including hypertension. The results of these tests do not necessarily indicate the presence of ipertensione secondaria, as the diagnosis of hypertension is based on a combination of symptoms and medical history.

2. A patient with hypertension who undergoes a stress test does not suggest the presence of ipertensione secondaria.

A stress test is a non-invasive diagnostic test that measures the response of the heart to physical exercise. The results of a stress test do not necessarily indicate the presence of ipertensione secondaria, as the diagnosis of hypertension is based on a combination of symptoms and medical history.

3. A patie

### Integrazione Validazione Esterna
Introduciamo un modulo di ricerca simulata per incrociare i dati generati dall'LLM con fonti bibliografiche verificate.

In [13]:
import requests

def mock_medical_web_search(query):
    """
    Simula una chiamata a un'API medica (es. PubMed) per validare le affermazioni.
    In un ambiente reale, qui si userebbe biopython o l'API di NCBI.
    """
    database = {
        "MSLT": "The Multiple Sleep Latency Test (MSLT) was developed by Mary Carskadon and William Dement in 1977, not MIT.",
        "metotrexato": "Methotrexate is a folate antagonist. Supplementation with folic acid reduces toxicity.",
        "ipertensione": "S4 gallop is a sign of reduced ventricular compliance, common in long-standing essential hypertension."
    }

    # Logica di matching semplice per la simulazione
    for key in database:
        if key.lower() in query.lower():
            return f"[FONTE VERIFICATA] {database[key]}"
    return "[WARNING] Nessun riscontro trovato nelle banche dati ufficiali."

class EnhancedManager(ManagerAgent):
    def validate_with_external_source(self, ai_response):
        self.log("Avvio validazione incrociata con database esterni...")
        validation_result = mock_medical_web_search(ai_response)
        return validation_result

def run_validated_analysis():
    manager = EnhancedManager()
    med_agent = AGENT_REGISTRY['medical_researcher']

    # Task critico: l'origine dell'MSLT (dove il modello ha precedentemente allucinato)
    query = "Chi ha inventato il test MSLT e in che anno?"
    ai_response = manager.llm.generate(f"{med_agent.system_prompt} {query}")

    external_validation = manager.validate_with_external_source(ai_response)

    print(f"\n--- Risposta AI ---\n{ai_response}")
    print(f"\n--- Validazione Esterna ---\n{external_validation}")

run_validated_analysis()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[*] Avvio validazione incrociata con database esterni...

--- Risposta AI ---
Sure, I can provide evidence-based medical insights based on current research and medical knowledge.

Chi ha inventato il test MSLT (Multiple Sclerosis Lesion Test) è una tecnologia medica che utilizza la radiografia per rilevare lesioni di sclerosi multiple (MS) in pazienti con malattia. Il test è stato sviluppato nel 2009 da un gruppo di ricercatori del Massachusetts Institute of Technology (MIT) e del National Institutes of Health (NIH).

Il test MSLT è stato approvato dal FDA (Food and Drug Administration) nel 2013 e nel 2014 è stato utilizzato per la prima volta in un'operazione diagnostica di MS.

Il test MSLT è stato sviluppato per migliorare la diagnosi e la prevenzione della malattia, in particolare per i pazienti con una storia di MS che non hanno risposto alle terapie standard.

Il test MSLT è stato testato su un gruppo di pazienti con MS e ha dimostrato una sensibilità e una specificità di 95% e 9

In [14]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.0 MB/s eta 0:00:00


In [15]:
from Bio import Entrez
import time

# Configurazione Entrez (NCBI richiede un'email per l'identificazione)
Entrez.email = "your_email@example.com"

def real_medical_search(query, max_results=1):
    """
    Esegue una ricerca reale su PubMed via Entrez.
    """
    try:
        # 1. Ricerca degli ID degli articoli
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record["IdList"]
        if not id_list:
            return "[NCBI] Nessun riscontro trovato per questa query."

        # 2. Recupero dei dettagli del primo articolo trovato
        handle = Entrez.efetch(db="pubmed", id=id_list[0], rettype="medline", retmode="text")
        details = handle.read()
        handle.close()

        # Estrazione semplificata del titolo (nella pratica si userebbe Medline.read)
        title_line = [line for line in details.split('\n') if line.startswith('TI  -')]
        title = title_line[0][6:] if title_line else "Titolo non trovato"

        return f"[FONTE REALE PUBMED] PMID: {id_list[0]} - Titolo: {title}"
    except Exception as e:
        return f"[ERRORE API] Impossibile connettersi a NCBI: {str(e)}"

class RealValidationManager(EnhancedManager):
    def validate_with_pubmed(self, ai_response):
        self.log("Contattando i server NCBI per la validazione reale...")
        # Estraiamo una keyword semplificata dalla risposta per la ricerca
        # In un caso reale useremmo un altro agente LLM per estrarre le keywords
        keywords = "Multiple Sleep Latency Test history" if "MSLT" in ai_response else "Medical research"
        return real_medical_search(keywords)

def test_real_validation():
    manager = RealValidationManager()
    # Esempio di risposta potenzialmente allucinata
    fake_ai_response = "Il test MSLT è stato inventato nel 1999 dal MIT."

    validation = manager.validate_with_pubmed(fake_ai_response)
    print(f"\n--- Validazione Reale via Biopython ---\n{validation}")

test_real_validation()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[*] Contattando i server NCBI per la validazione reale...

--- Validazione Reale via Biopython ---
[FONTE REALE PUBMED] PMID: 41552052 - Titolo: Stiff Person Syndrome: A Case Report.


In [17]:
def extract_pmids(search_term, max_results=5):
    """
    Estrae automaticamente una lista di PMID per un dato termine di ricerca.
    """
    try:
        handle = Entrez.esearch(db="pubmed", term=search_term, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()
        return record.get("IdList", [])
    except Exception as e:
        print(f"Errore durante l'estrazione PMID: {e}")
        return []

# Test rapido della funzione
termini = "asthma pediatric nocturnal symptoms"
lista_pmid = extract_pmids(termini)
print(f"PMID trovati per '{termini}': {lista_pmid}")

PMID trovati per 'asthma pediatric nocturnal symptoms': ['42017769', '41727422', '41583298', '41234762', '41219248']


In [18]:
def fetch_abstracts(pmid_list):
    """
    Recupera gli abstract completi per una lista di PMID.
    """
    if not pmid_list:
        return "Nessun PMID fornito."

    try:
        # Uniamo i PMID in una stringa separata da virgole
        ids = ",".join(pmid_list)
        handle = Entrez.efetch(db="pubmed", id=ids, retmode="xml")
        records = Entrez.read(handle)
        handle.close()

        results = []
        for article in records['PubmedArticle']:
            title = article['MedlineCitation']['Article']['ArticleTitle']
            # L'abstract può essere diviso in più sezioni
            abstract_data = article['MedlineCitation']['Article'].get('Abstract', {})
            abstract_text = " ".join(abstract_data.get('AbstractText', ["Abstract non disponibile"]))

            results.append({
                "title": title,
                "abstract": abstract_text
            })
        return results
    except Exception as e:
        return f"Errore durante il recupero degli abstract: {e}"

# Test con i primi 2 PMID trovati nel passo precedente
test_abstracts = fetch_abstracts(lista_pmid[:2])
for i, art in enumerate(test_abstracts):
    print(f"--- Articolo {i+1} ---")
    print(f"TITOLO: {art['title']}")
    print(f"ABSTRACT: {art['abstract'][:300]}...\n")

--- Articolo 1 ---
TITOLO: Association between nocturnal asthma, symptom severity and clinical outcomes: a systematic review.
ABSTRACT: Nocturnal asthma is characterized by the worsening of asthma symptoms during the night and indicates more severe disease. This review aims to analyze evidence regarding the association between nocturnal asthma, symptom severity, pulmonary function and clinical outcomes in both adult and pediatric po...

--- Articolo 2 ---
TITOLO: Subglottic inflammatory myofibroblastic tumor: A rare case report in a pediatric patient.
ABSTRACT: <b>Background:</b> Inflammatory myofibroblastic tumour (IMT) of the pediatric laryngotracheal complex is exceedingly rare, with fewer than nine subglottic IMT cases reported in the literature up to 2025. We present an additional case and highlight the importance of early recognition. <b>Case present...



In [16]:
def solve_asthma_query():
    manager = RealValidationManager()
    med_agent = AGENT_REGISTRY['medical_researcher']

    question = """Quale sintomo caratteristico dell'asma infantile tende a peggiorare durante la notte o al risveglio?
Options: Cefalee, Febbre alta, Eruzione cutanea, Tosse notturna irritativa non produttiva, Dolore toracico persistente"""

    # 1. Generazione risposta AI
    prompt = f"{med_agent.system_prompt}\n\nAnalizza la domanda e identifica l'opzione corretta:\n{question}"
    ai_response = manager.llm.generate(prompt, max_tokens=200)

    # 2. Validazione reale via PubMed
    # Cerchiamo specificamente 'nocturnal cough asthma children'
    search_query = "nocturnal cough asthma children characteristics"
    validation = real_medical_search(search_query)

    print(f"--- Analisi Medica Agente ---\n{ai_response}")
    print(f"\n--- Validazione Scientifica (PubMed) ---\n{validation}")

solve_asthma_query()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- Analisi Medica Agente ---
L'opzione corretta è: Dolore toracico persistente.

Dolore toracico persistente è un sintomo caratteristico dell'asma infantile che peggiora durante la notte o al risveglio. Questo sintomo è spesso associato a febbre alta, cefalee, eccessiva eruzione cutanea, e dolore toracico persistente non produttivo. Questo sintomo è un sintomo di disturbo dell'apparato respiratorio, che può essere causato da un'infezione o da un'irritazione del tessuto toracico. Questo sintomo può essere più frequente in bambini che hanno un'infezione dell'asma, come in caso di infezione da Streptococcus pneumoniae.

Inoltre, il dolore toracico persistente può

--- Validazione Scientifica (PubMed) ---
[FONTE REALE PUBMED] PMID: 37371263 - Titolo: Residual Cough and Asthma-like Symptoms Post-COVID-19 in Children.


In [19]:
def summarize_abstracts(abstract_list):
    """
    Sintetizza una lista di abstract medici utilizzando il modello LLM locale.
    """
    manager = RealValidationManager()
    summaries = []

    print(f"[*] Inizio sintesi di {len(abstract_list)} abstract...")

    for i, item in enumerate(abstract_list):
        title = item['title']
        text = item['abstract']

        # Prompt per la sintesi focalizzata sui punti chiave
        prompt = f"""Sintetizza il seguente abstract medico in due frasi, evidenziando i risultati principali:
        TITOLO: {title}
        ABSTRACT: {text}

        Sintesi:"""

        summary = manager.llm.generate(prompt, max_tokens=150)
        summaries.append({
            "title": title,
            "summary": summary
        })
        print(f"[*] Completata sintesi {i+1}/{len(abstract_list)}")

    return summaries

# Test della funzione di sintesi con gli abstract precedentemente recuperati
if 'test_abstracts' in globals():
    sintesi_risultati = summarize_abstracts(test_abstracts)

    print("\n--- REPORT SINTETICO DEGLI STUDI ---")
    for s in sintesi_risultati:
        print(f"\nSTUDIO: {s['title']}")
        print(f"SINTESI LLM: {s['summary']}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[*] Inizio sintesi di 2 abstract...
[*] Completata sintesi 1/2
[*] Completata sintesi 2/2

--- REPORT SINTETICO DEGLI STUDI ---

STUDIO: Association between nocturnal asthma, symptom severity and clinical outcomes: a systematic review.
SINTESI LLM: In un'abstract medico, si evidenzia il risultato principale di questa ricerca sistematica: la relazione tra la malattia di asthma nocturale e la severità dei sintomi durante la notte, con risultati più severi per la malattia. Questo studio si concentra su studi che comparano la severità dei sintomi di asthma durante la notte, la funzionalità pulmonare e i risultati clinici in popolazioni adulte e pediatrica. Le fonti di ricerca sono Embase, PubMed, CINAHL, Cochrane (Wiley), Scopus, e LILACS. Si sono incl

STUDIO: Subglottic inflammatory myofibroblastic tumor: A rare case report in a pediatric patient.
SINTESI LLM: Il titolo del documento è "Subglottic inflammatory myofibroblastic tumor: Un caso raro in un pediatra" e si riferisce a un caso d

In [20]:
import pandas as pd

def export_summaries_to_csv(summaries, filename="medical_summaries.csv"):
    """
    Esporta i risultati della sintesi in un file CSV.
    """
    if not summaries:
        print("[!] Nessun dato da esportare.")
        return

    df = pd.DataFrame(summaries)
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"[*] Risultati esportati con successo in: {filename}")
    display(df)

# Esecuzione dell'esportazione se i risultati sono presenti
if 'sintesi_risultati' in globals():
    export_summaries_to_csv(sintesi_risultati)

[*] Risultati esportati con successo in: medical_summaries.csv


,title,summary
0,"Association between nocturnal asthma, symptom ...","In un'abstract medico, si evidenzia il risulta..."
1,Subglottic inflammatory myofibroblastic tumor:...,"Il titolo del documento è ""Subglottic inflamma..."


In [22]:
def solve_virus_comparison_fixed():
    manager = RealValidationManager()
    med_agent = AGENT_REGISTRY['medical_researcher']

    # Ottimizzazione query: cerchiamo termini specifici per validare l'opzione 4
    search_query = "Rotavirus seasonal Norovirus epidemic patterns"
    pubmed_validation = real_medical_search(search_query)

    print(f"--- Validazione Scientifica (PubMed) ---\n{pubmed_validation}")

    print("\n--- Analisi Corretta ---")
    print("L'opzione corretta è la 4: Rotavirus (andamento stagionale) e Norovirus (andamento epidemico).")
    print("- Rotavirus: causa principale di gastroenterite nei bambini <5 anni, con picchi stagionali (inverno/primavera).")
    print("- Norovirus: altamente contagioso, causa frequenti focolai epidemici in contesti chiusi (navi, scuole) indipendentemente dalla stagione.")
    print("- Note sui diametri: Rotavirus (~70nm), Norovirus (~27-35nm), rendendo errata l'opzione 2.")

solve_virus_comparison_fixed()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- Validazione Scientifica (PubMed) ---
[FONTE REALE PUBMED] PMID: 41443693 - Titolo: [Etiology and epidemiological characteristics of viral diarrhea in Guangzhou, 

--- Analisi Corretta ---
L'opzione corretta è la 4: Rotavirus (andamento stagionale) e Norovirus (andamento epidemico).
- Rotavirus: causa principale di gastroenterite nei bambini <5 anni, con picchi stagionali (inverno/primavera).
- Norovirus: altamente contagioso, causa frequenti focolai epidemici in contesti chiusi (navi, scuole) indipendentemente dalla stagione.
- Note sui diametri: Rotavirus (~70nm), Norovirus (~27-35nm), rendendo errata l'opzione 2.
